In [1]:
from pathlib import Path
import numpy as np
import sys

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from spike_classifier.annotate_spikes import annotate_spikes
from spike_classifier.train_classifier import train_spike_classifier
from spike_classifier.prepare_data import prepare_spike_data
from utils.label_utils import get_label_value, reset_spike_labels, compute_data_summary
from classifier_pipeline.verbose_utils import print_data_summary


In [2]:

ROI_DATA_PATH = PROJECT_ROOT / "data" / "all_roi_features.npy"
assert ROI_DATA_PATH.exists(), f"Data file not found: {ROI_DATA_PATH}"

MODEL_OUT_DIR = PROJECT_ROOT / "models"
MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = PROJECT_ROOT / "config" / "classifier_config.yaml"

print(f"Path to data: {ROI_DATA_PATH}")
print(f"Saving models to: {MODEL_OUT_DIR}")
print(f"Config: {CONFIG_PATH}")

Path to data: C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\data\all_roi_features.npy
Saving models to: C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\models
Config: C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\config\classifier_config.yaml


In [3]:

reset = False # Gated purposely to avoid accidental resets
if reset:
    roi_dict = np.load(ROI_DATA_PATH, allow_pickle=True).item()
    roi_dict, n_reset = reset_spike_labels(roi_dict)
    np.save(ROI_DATA_PATH, roi_dict, allow_pickle=True)
    print(f"Reset {n_reset} labels to unlabeled")

In [4]:
roi_dict = prepare_spike_data(
    input_path=str(ROI_DATA_PATH),
    output_path=None,  
    max_rois=None      
)


Loaded 18652 ROIs from C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\data\all_roi_features.npy
Processed 573 good ROIs
Skipped 18079 bad ROIs
Preserved 1368 existing labels
Saved to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\data\all_roi_features.npy

  ROI Summary
  Total rois: 18652
  Good: 573 | Bad: 280 | Unlabeled: 17799
  Manual: 853 | Auto: 0
  Total spikes stored: 10329



In [5]:


N_ANNOTATIONS = 5000

annotate_spikes(
    data_path=ROI_DATA_PATH,
    unlabeled_only=True,
)



Loaded 18652 ROIs from C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\data\all_roi_features.npy
Session ended by user. Saving progress...
Saved to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\data\all_roi_features.npy
Saved to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\data\all_roi_features.npy

  SPIKE Annotation Summary
  Total:     557
  Labeled:   0
  Updated:   0
  Confirmed: 0
  Skipped:   0


  SPIKE Summary
  ROIs: 18652  (573 with spikes)
  Total spikes: 10329
  Good: 358 | Bad: 1010 | Unlabeled: 8961
  Manual: 1368 | Auto: 0



{'total': 557,
 'labeled': 0,
 'updated': 0,
 'confirmed': 0,
 'skipped': 0,
 'level': 'spike'}

In [6]:

name = "spike_classifier" # TODO Change as desired for your organizational needs

results = train_spike_classifier(
    config_path=CONFIG_PATH,
    data_path=ROI_DATA_PATH,
    name=name,
    output_dir=MODEL_OUT_DIR,
    verbose=True,
    manual_only=True
)


Dataset Summary
--------------------------------------------------
Total labeled datapoints: 1368
  Train: 1094 | Test: 274

Label distribution:
              Bad (0)  Good (1)
  Train           813       281
  Test            197        77
  Total          1010       358

Training on: Manual labels only


c:\Users\mzinn1\AppData\Local\anaconda3\envs\gcamp\lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



--------------------------------------------------
TUNED MODEL SUMMARY
--------------------------------------------------
Model:     LogisticRegression
Transform: sqrt
Features:  ['spike_prom', 'mini_prom', 'distance']

Hyperparameters:
  C: 10
  class_weight: None
  max_iter: 200
  penalty: l1
  solver: saga

Metrics:
  CV Accuracy:   0.9634
  Test Accuracy: 0.9635
  ROC AUC:       0.9790
  F1:            0.9632
  Precision:     0.9634
  Recall:        0.9635

Confusion Matrix:
              Pred 0  Pred 1
  Actual 0    194     3      
  Actual 1    7       70     
--------------------------------------------------
Saved model to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\models\spike_classifier.joblib
Saved results to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\models\spike_classifier_results.json
Saved results to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\models


c:\Users\mzinn1\AppData\Local\anaconda3\envs\gcamp\lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
